# MASTER clean - all years 2011-2026 (v2)

Run top to bottom, once. Order of the stages:

1. **Load + the 2026 name fixes + language count** (your cells, unchanged in logic)
2. **Salary** - parse `CompTotal` text, annualise, FX to USD for *that survey year*, one final column `Comp_USD_Annual`
3. **Population filter** - no students, no hobbyists, no unemployed, then no usable salary
4. **Cutoff** - plausibility window on `Comp_USD_Annual`
5. **Text columns** - normalise `;` lists, collapse spelling variants, vocabulary reports
6. **Keep columns + save**

Every row-dropping step prints how many rows it removes, per rule and per year, so nothing disappears silently. All the patterns used to decide "student / hobbyist / unemployed" are in one `CONFIG` cell and the cell prints the exact values each pattern matched.

**Things that changed vs your v1 notebook:** the old cells 9-11 and 13 (exploration) are folded into the diagnostics; cells 14-15 (`pd.to_numeric(df['CompTotal'])`) are removed because they turn every band (`<$10,000`, `$20,000 - $40,000`, ...) into NaN; cell 12 no longer reloads the CSV (it threw away the fixes above it); `survey_year` and `uid` stay in the kept columns because your final assert needs them.

## 1. Load and fix the 2026 names

In [9]:
import re
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

SRC = "combined/all_years.csv"
# keep_default_na=False: real answers such as "None" are not silently turned into NaN; only truly empty cells are NaN
df = pd.read_csv(SRC, low_memory=False, keep_default_na=False, na_values=[""])
print(df.shape)
row_log = [("loaded", len(df), len(df))]
def log_rows(step, before, after):
    row_log.append((step, before, after))
    print("[ROWS] %-52s %7d -> %7d  (removed %d)" % (step, before, after, before - after))

(417467, 160)


In [10]:
# wrong name (only in 2026) -> correct name used by the other years
MERGE = {
    "LanguagesHaveWorkedWith":    "LanguageHaveWorkedWith",
    "LanguagesWantToWorkWith":    "LanguageWantToWorkWith",
    "DatabasesHaveWorkedWith":    "DatabaseHaveWorkedWith",
    "DatabasesWantToWorkWith":    "DatabaseWantToWorkWith",
    "Education":                  "EdLevel",
    "CloudPlatHaveWorkedWith":    "PlatformHaveWorkedWith",
    "CloudPlatWantToWorkWith":    "PlatformWantToWorkWith",
    "WebFrameworkHaveWorkedWith": "WebframeHaveWorkedWith",
    "WebFrameworkWantToWorkWith": "WebframeWantToWorkWith",
}
for wrong, right in MERGE.items():
    if wrong not in df.columns:
        print(f"skip: '{wrong}' not in file"); continue
    if right not in df.columns:
        df = df.rename(columns={wrong: right}); print(f"renamed {wrong} -> {right}"); continue
    both = df[wrong].notna() & df[right].notna()
    if both.any():
        print(f"WARNING {wrong}/{right}: {both.sum()} rows have values in both, keeping '{right}'")
    df[right] = df[right].fillna(df[wrong])
    df = df.drop(columns=wrong)
    print(f"merged {wrong} -> {right}")

# drop the columns kept by mistake in 2026
DROP = ["TechEndorseIntro", "SOTagsWant Entry", "CommPlatformHaveEntr", "CommPlatformWantEntr"]
df = df.drop(columns=[c for c in DROP if c in df.columns])
if "LearnCodeHow" in df.columns:
    df = df.rename(columns={"LearnCodeHow": "LearnCode__LearnCodeHow"})

# number of items in the ";" list; blank answer stays blank (not 0)
df["LanguageHaveWorkedWith_count"] = df["LanguageHaveWorkedWith"].str.split(";").str.len().astype("Int64")
print(df.shape)

merged LanguagesHaveWorkedWith -> LanguageHaveWorkedWith
merged LanguagesWantToWorkWith -> LanguageWantToWorkWith
merged DatabasesHaveWorkedWith -> DatabaseHaveWorkedWith
merged DatabasesWantToWorkWith -> DatabaseWantToWorkWith
merged Education -> EdLevel
merged CloudPlatHaveWorkedWith -> PlatformHaveWorkedWith
merged CloudPlatWantToWorkWith -> PlatformWantToWorkWith
merged WebFrameworkHaveWorkedWith -> WebframeHaveWorkedWith
merged WebFrameworkWantToWorkWith -> WebframeWantToWorkWith
(417467, 147)


## 2. Salary

**Order (mandatory, from your notes):** parse the raw text -> annualise in local currency -> USD. Midpoint rules are fixed: `"$A - $B"` -> (A+B)/2, `"<$X"` -> X/2, `">$X"` -> X + 10,000.

### 2.1 Parse `CompTotal` (raw text, never `pd.to_numeric` first)
One deviation from "strip `$` and `,` everywhere": a plain number such as `0,00` or `50.000,00` uses a **decimal comma**, and deleting the comma would turn `50.000,00` into 50. So `,` is only treated as a thousands separator when it really is one (`1,234` / `1,234,567.5`); anything ambiguous becomes NaN and is counted below, not guessed.

In [11]:
LOWER_RULE = lambda cap: cap / 2        # "<$10,000"  -> 5,000
UPPER_BUMP = 10_000                      # ">$200,000" -> 210,000

_PLAIN  = re.compile(r"^\d+(\.\d+)?([eE][+-]?\d+)?$|^\.\d+$")
_THOUS  = re.compile(r"^\d{1,3}(,\d{3})+(\.\d+)?$")

def _num(t):
    t = t.replace("$", "").strip()
    if _PLAIN.match(t):
        return float(t)
    if _THOUS.match(t):
        return float(t.replace(",", ""))
    raise ValueError(t)

def extract_midpoint(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip()
    if not s or s.lower() == "nan":
        return np.nan
    try:
        if s.startswith("<"):
            return LOWER_RULE(_num(s[1:]))
        if s.startswith(">"):
            return _num(s[1:]) + UPPER_BUMP
        m = re.fullmatch(r"\$?([\d,]+)\s*-\s*\$?([\d,]+)", s)           # band "$20,000 - $40,000"
        if m:
            return (_num(m.group(1)) + _num(m.group(2))) / 2
        return _num(s)                                                    # plain number
    except ValueError:
        return np.nan

for c, want in [("<$10,000", 5000), ("<$20,000", 10000), (">$140,000", 150000), (">$160,000", 170000), (">$200,000", 210000),
                ("$20,000 - $40,000", 30000), ("100000.0", 100000), ("$80,000", 80000), ("1,234,567.5", 1234567.5)]:
    assert extract_midpoint(c) == want, (c, extract_midpoint(c))
assert np.isnan(extract_midpoint("0,00")) and np.isnan(extract_midpoint("50.000,00")) and np.isnan(extract_midpoint("abc"))
print("midpoint rules verified on the spot-check values")

raw = df["CompTotal"]
df["CompTotal_Num"] = raw.map(extract_midpoint)                           # CompTotal itself is NOT overwritten
fmt = np.select([raw.isna(), raw.astype(str).str.startswith("<"), raw.astype(str).str.startswith(">"), raw.astype(str).str.contains(" - ", regex=False)],
                ["blank", "lower threshold", "upper threshold", "range"], default="plain number")
tab = pd.crosstab(pd.Series(fmt, index=df.index, name="format"), df["CompTotal_Num"].notna().map({True: "parsed", False: "NaN"}))
print(tab)
unparsed = raw.notna() & df["CompTotal_Num"].isna()
print("non-blank CompTotal that could not be parsed:", int(unparsed.sum()))
print(raw[unparsed].value_counts().head(10))

midpoint rules verified on the spot-check values
CompTotal_Num      NaN  parsed
format                        
blank            12891       0
lower threshold      0   14394
plain number        70  332034
range                0   56097
upper threshold      0    1981
non-blank CompTotal that could not be parsed: 70
CompTotal
8000,00     4
4000,00     4
2500,00     2
1000,00     2
3500,00     2
500,00      2
2688,82     1
35000,00    1
7080,65     1
3190,00     1
Name: count, dtype: int64


### 2.2 Annualise in local currency
`Monthly` x12, `Weekly` x52, `Yearly` x1, **blank = already annual** (x1).

In [12]:
print("CompFreq:", df["CompFreq"].value_counts(dropna=False).to_dict())
MULT = {"Yearly": 1, "Monthly": 12, "Weekly": 52}
unknown_freq = sorted(set(df["CompFreq"].dropna()) - set(MULT))
assert not unknown_freq, "unknown CompFreq values: " + str(unknown_freq)
df["CompTotal_Annual_Local"] = df["CompTotal_Num"] * df["CompFreq"].map(MULT).fillna(1)

CompFreq: {nan: 200180, 'Yearly': 118808, 'Monthly': 91849, 'Weekly': 6630}


### 2.3 Currency -> ISO code (merge key)
Built from the 414 distinct values in your own list. They come in four shapes, so one rule + one small table covers everything:

| shape | years | example | rule |
|---|---|---|---|
| ISO + tab + name | 2021-2024 | `INR\tIndian rupee` | first 3 capitals |
| ISO + space + name | 2025-2026 | `INR Indian rupee` | first 3 capitals |
| ISO only | 2019-2020 | `INR` | first 3 capitals |
| name + symbol | 2017-2018 | `Indian rupees (₹)`, `Euros (€)`, `Swiss francs` | exact table below (21 names) |

`Bitcoin (btc)` (64 rows) is not a currency with a USD rate, so it stays NaN; `XDR` (SDR) is kept as is and will simply have no rate. 2011-2016 pay is in USD bands and has no `Currency` at all, so blank currency in those years = `USD`.

In [13]:
NAME_2017_2018 = {
    "U.S. dollars ($)": "USD", "Euros (€)": "EUR", "British pounds sterling (£)": "GBP", "Indian rupees (₹)": "INR", "Indian rupees (?)": "INR",
    "Canadian dollars (C$)": "CAD", "Australian dollars (A$)": "AUD", "Brazilian reais (R$)": "BRL", "Russian rubles (₽)": "RUB", "Russian rubles (?)": "RUB",
    "Polish złoty (zł)": "PLN", "Polish zloty (zl)": "PLN", "Swedish kroner (SEK)": "SEK", "Swiss francs": "CHF", "South African rands (R)": "ZAR",
    "Mexican pesos (MXN$)": "MXN", "Danish krone (kr)": "DKK", "Norwegian krone (kr)": "NOK", "Chinese yuan renminbi (¥)": "CNY",
    "Japanese yen (¥)": "JPY", "Singapore dollars (S$)": "SGD",
}
NOT_FIAT = {"Bitcoin (btc)"}
_ISO = re.compile(r"^([A-Z]{3})(?:[\t ]|$)")

def currency_iso(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s in NAME_2017_2018:
        return NAME_2017_2018[s]
    if s in NOT_FIAT:
        return np.nan
    m = _ISO.match(s)
    return m.group(1) if m else np.nan

cur_vals = df["Currency"].dropna().unique()
cmap = pd.Series({v: currency_iso(v) for v in cur_vals})
print("distinct Currency values:", len(cur_vals), "| mapped:", int(cmap.notna().sum()), "| NOT mapped:", int(cmap.isna().sum()))
print("not mapped:", sorted(cmap[cmap.isna()].index))
assert cmap[cmap.isna()].index.isin(NOT_FIAT).all(), "unexpected Currency text that is neither ISO-prefixed nor in the 2017-2018 table"
# two raw spellings of one currency in the same year would be fine; a raw value mapping to two codes cannot happen by construction
df["Currency_ISO"] = df["Currency"].map(currency_iso)
df.loc[df["Currency_ISO"].isna() & df["Currency"].isna() & (df["survey_year"] <= 2016), "Currency_ISO"] = "USD"
print("Currency_ISO blank by year (should be only 2017+ rows without a currency):")
print(df.loc[df["Currency_ISO"].isna()].groupby("survey_year").size().to_dict())
print("distinct ISO codes:", df["Currency_ISO"].nunique())
cmap.rename("iso").to_csv("currency_map.csv")

distinct Currency values: 414 | mapped: 413 | NOT mapped: 1
not mapped: ['Bitcoin (btc)']
Currency_ISO blank by year (should be only 2017+ rows without a currency):
{2017: 906, 2018: 4586}
distinct ISO codes: 150


### 2.4 Which rows actually need an exchange rate?
`ConvertedCompYearly` is the survey's own USD figure (already annual, already converted with *that year's* rate). Only rows that have a local amount but **no** `ConvertedCompYearly` need our own FX, so the cell lists exactly those (year x currency) pairs and saves them to `fx_needed.csv`. That is the whole external-data shopping list.

In [14]:
has_local = df["CompTotal_Annual_Local"].notna()
has_usd = df["ConvertedCompYearly"].notna()
need = df.loc[has_local & ~has_usd & (df["Currency_ISO"] != "USD")]
fx_needed = need.groupby(["survey_year", "Currency_ISO"], dropna=False).size().rename("n_rows").reset_index().sort_values("n_rows", ascending=False)
fx_needed.to_csv("fx_needed.csv", index=False)
print("rows with a local amount, no ConvertedCompYearly, non-USD currency:", len(need))
print("  of those with no currency code at all:", int(need["Currency_ISO"].isna().sum()))
print("by year:", need.groupby("survey_year").size().to_dict())
print(fx_needed.head(25).to_string(index=False))
print("\nrows with CompTotal blank but ConvertedCompYearly present (rescued, already USD):", int((df["CompTotal_Num"].isna() & has_usd).sum()),
      "| by year:", df.loc[df["CompTotal_Num"].isna() & has_usd].groupby("survey_year").size().to_dict())

rows with a local amount, no ConvertedCompYearly, non-USD currency: 0
  of those with no currency code at all: 0
by year: {}
Empty DataFrame
Columns: [survey_year, Currency_ISO, n_rows]
Index: []

rows with CompTotal blank but ConvertedCompYearly present (rescued, already USD): 12961 | by year: {2017: 12891, 2018: 70}


### 2.5 Exchange-rate table: `local units per 1 USD`, one value per (survey year, currency)
Two sources, in this order:

1. **Survey-implied rate** = median of `local_annual / ConvertedCompYearly` over rows that have both, per year and currency. This is the rate the survey itself used for *that* year, it merges on the same normalised code, and for the big currencies it rests on thousands of rows. It is only trusted when `n >= MIN_N` and the spread of the ratio is tight (`q75/q25 <= MAX_SPREAD`); a bad group (a few rows, or a currency mix-up like UAH 2026 in your earlier output) is rejected, not used.
2. **External annual-average table** (optional file `fx_external.csv`, columns `survey_year,iso,local_per_usd`, convention **local per USD**). Fills the pairs the survey cannot give and is used to *check* the survey rates where both exist. See the note under the cell for where to get it.

A row that still has no rate stays NaN (never guessed).

In [15]:
MIN_N, MAX_SPREAD = 5, 1.05
both = has_local & has_usd & df["Currency_ISO"].notna() & (df["ConvertedCompYearly"] > 0) & (df["CompTotal_Annual_Local"] > 0)
ratio = (df.loc[both, "CompTotal_Annual_Local"] / df.loc[both, "ConvertedCompYearly"])          # local per USD
g = ratio.groupby([df.loc[both, "survey_year"], df.loc[both, "Currency_ISO"]])
implied = pd.DataFrame({"local_per_usd": g.median(), "n_rows": g.size(), "spread": g.quantile(.75) / g.quantile(.25)})
implied["trusted"] = (implied["n_rows"] >= MIN_N) & (implied["spread"] <= MAX_SPREAD)
implied.loc[implied.index.get_level_values("Currency_ISO") == "USD", ["local_per_usd", "trusted"]] = [1.0, True]
print("survey-implied rates:", len(implied), "| trusted:", int(implied["trusted"].sum()), "| rejected:", int((~implied["trusted"]).sum()))
print("rejected groups with the most rows:")
print(implied[~implied["trusted"]].sort_values("n_rows", ascending=False).head(8))

EXT = Path("fx_external.csv")
if EXT.exists():
    ext = pd.read_csv(EXT)
    ext["iso"] = ext["iso"].astype(str).str.strip().str.upper()
    ext = ext.drop_duplicates(["survey_year", "iso"]).set_index(["survey_year", "iso"])["local_per_usd"]
    ext.index.names = ["survey_year", "Currency_ISO"]
    chk = implied.join(ext.rename("external"), how="inner")
    chk["pct_diff"] = chk["local_per_usd"] / chk["external"] - 1
    chk.to_csv("fx_check.csv")
    big = chk[chk["n_rows"] >= 50]
    print("\nexternal file found:", len(ext), "rates | pairs compared with the survey (n>=50):", len(big),
          "| differ by more than 10%:", int((big["pct_diff"].abs() > 0.10).sum()))
    print(big.reindex(big["pct_diff"].abs().sort_values(ascending=False).index).head(10)[["local_per_usd", "external", "pct_diff", "n_rows"]])
else:
    ext = pd.Series(dtype=float, name="local_per_usd")
    ext.index = pd.MultiIndex.from_arrays([[], []], names=["survey_year", "Currency_ISO"])
    print("\nno fx_external.csv next to the notebook: survey-implied rates only (the check against an outside source is skipped)")

rate = implied.loc[implied["trusted"], "local_per_usd"]
fx = pd.concat([rate.rename("survey"), ext.rename("external")], axis=1)
fx["local_per_usd"] = fx["survey"].fillna(fx["external"])          # survey first, external fills the gaps
fx["fx_source"] = np.where(fx["survey"].notna(), "survey", "external")
fx.to_csv("fx_rates_used.csv")
print("final rate table:", len(fx), "rows | from survey:", int((fx["fx_source"] == "survey").sum()), "| from external:", int((fx["fx_source"] == "external").sum()))

survey-implied rates: 1022 | trusted: 736 | rejected: 286
rejected groups with the most rows:
                          local_per_usd  n_rows     spread  trusted
survey_year Currency_ISO                                           
2026        UAH               44.965254     185  44.966373    False
2019        VES             3466.666667       7   1.056738    False
            RWF              884.955752       4   1.000316    False
2025        MVR               15.405068       4   1.000008    False
            MZN               63.959066       4   1.000033    False
2026        AOA              918.082958       4   1.002819    False
2025        BYN                3.270595       4   1.000022    False
2020        GIP                0.773638       4   1.000026    False

no fx_external.csv next to the notebook: survey-implied rates only (the check against an outside source is skipped)
final rate table: 736 rows | from survey: 736 | from external: 0


**Where to get `fx_external.csv`.** Only needed for the pairs listed in `fx_needed.csv` that the survey rate cannot cover, so look at that list first (2.4). The best free source for an annual *average* by year is the World Bank / IMF series "Official exchange rate (LCU per US$, period average)", indicator `PA.NUS.FCRF`. It is keyed by **country**, not currency, so it needs a currency -> country map, and two traps: euro-area countries are listed in their *legacy* currency (Austria 2023 shows about 12.7, not about 0.92), so take EUR from the survey or from the ECB, never from a member country; and official rates can sit far from the rate people actually use (IRR, ARS, NGN, EGP, ...). `get_fx_worldbank.py` (separate file) downloads it and writes the file in the right shape. The IRS yearly-average table or the ECB / Fed H.10 averages work too, as long as you write `survey_year, iso, local_per_usd`.

### 2.6 The one salary column
`comp_source` says how each value was made:

- `survey_usd` - `ConvertedCompYearly` (CompTotal also present)
- `rescued_usd` - `CompTotal` blank but `ConvertedCompYearly` present (already USD, bypasses FX; mostly 2017)
- `parsed_fx` - our own chain: parsed + annualised local amount / that year's rate
- `missing` - nothing usable

`PREFER_SURVEY_USD = True` makes the survey's figure win whenever it exists (it is the only figure that covers every year the same way); our own chain is still computed for every row as `Comp_USD_chain` so you can compare the two. Set it to `False` to use our chain wherever it can be computed.

In [16]:
PREFER_SURVEY_USD = True

idx = pd.MultiIndex.from_frame(df[["survey_year", "Currency_ISO"]])
df["FX_local_per_usd"] = fx["local_per_usd"].reindex(idx).values
df.loc[df["Currency_ISO"] == "USD", "FX_local_per_usd"] = 1.0
df["Comp_USD_chain"] = df["CompTotal_Annual_Local"] / df["FX_local_per_usd"]          # local per USD -> DIVIDE

surv = df["ConvertedCompYearly"]
chain = df["Comp_USD_chain"]
if PREFER_SURVEY_USD:
    df["Comp_USD_Annual"] = surv.where(surv.notna(), chain)
    df["comp_source"] = np.select([surv.notna() & df["CompTotal_Num"].notna(), surv.notna(), chain.notna()], ["survey_usd", "rescued_usd", "parsed_fx"], "missing")
else:
    df["Comp_USD_Annual"] = chain.where(chain.notna(), surv)
    df["comp_source"] = np.select([chain.notna(), surv.notna()], ["parsed_fx", "rescued_usd"], "missing")

df["CompIsTotalComp"] = (df["survey_year"] >= 2019).astype(int)       # 2011-2018 base salary only, 2019+ total compensation
print(pd.crosstab(df["survey_year"], df["comp_source"]))
no_rate = has_local & ~has_usd & df["FX_local_per_usd"].isna()
print("\nrows with an amount but no exchange rate (left NaN):", int(no_rate.sum()))
if no_rate.any():
    print(df.loc[no_rate].groupby(["survey_year", "Currency_ISO"], dropna=False).size().sort_values(ascending=False).head(10))

# own chain vs survey figure, where both exist (needs the external table to be a real test, otherwise it is circular)
cmp_ = df.loc[surv.notna() & chain.notna() & (surv > 0)].copy()
cmp_["r"] = cmp_["Comp_USD_chain"] / cmp_["ConvertedCompYearly"]
print("\nown chain / survey figure, per year (median and share within +-5%):")
print(cmp_.groupby("survey_year")["r"].agg(n="size", median="median", within_5pct=lambda s: (s.sub(1).abs() < 0.05).mean()).round(3))

comp_source  parsed_fx  rescued_usd  survey_usd
survey_year                                    
2011              2150            0           0
2012              3740            0           0
2013              5360            0           0
2014              4623            0           0
2015                 0            0       14857
2016                 0            0       41742
2017                 0        12891           0
2018                 0           70       47632
2019                 0            0       55823
2020                 0            0       34756
2021                 0            0       46844
2022                 0            0       38071
2023                 0            0       48019
2024                 0            0       23435
2025                 0            0       23947
2026                 0            0       13507

rows with an amount but no exchange rate (left NaN): 0

own chain / survey figure, per year (median and share within +-5%):
           

## 3. Population filter: no students, no hobbyists, no unemployed, no usable salary
All the patterns are in `CONFIG`. Each rule works on the **items** of a `;` list (`Employment`, `DevType` and `MainBranch` can hold several answers), is case-insensitive, and only fires when the column is filled, so a year that never asked a question (e.g. `MainBranch` before 2018) is not penalised. The first cell below prints every distinct value each pattern matched, so you can see exactly which answers are being dropped before anything is removed.

Notes on the choices:
- `Hobbyist` (the yes/no column, "do you code as a hobby?") is **not** used: a professional who also codes at weekends answers Yes, so dropping it would remove most professionals. The hobbyist rule is `MainBranch` "I code primarily as a hobby".
- `Student`: dropped if **any** item says student (`STRICT_STUDENT = True`), also in `DevType` and `OrgSize`. Set it to `False` to keep a student who also lists a job.
- `Employment`: a row is dropped when an item says student or not employed/retired and no item is a working status (`I prefer not to say` alone is unknown, so it is kept). Years with no `Employment` question cannot be checked and are kept (counted below).

In [17]:
CONFIG = {
    "MainBranch": {"drop": r"student|learning to code|primarily as a hobby|hobby"},          # student / learner / hobbyist
    "Employment_working": r"employed,? full|employed,? part|independent contractor|freelanc|self-employed|^employed$",
    "Employment_student": r"student",
    "Employment_notworking": r"not employed|unemploy|retired",
    "DevType": {"drop": r"student|unemploy|not employed|retired|hobby"},
    "OrgSize": {"drop": r"^student|not working|unemploy"},
}
STRICT_STUDENT = True

def explode_items(col):
    s = df[col].dropna().astype(str).str.split(";").explode().str.strip()
    return s[s != ""]

def row_any(col, pattern):
    ex = explode_items(col)
    hit = ex.str.contains(pattern, case=False, regex=True)
    return hit.groupby(level=0).any().reindex(df.index, fill_value=False)

def show_matches(col, pattern, label):
    ex = explode_items(col)
    m = ex[ex.str.contains(pattern, case=False, regex=True)]
    t = pd.crosstab(m.values, df.loc[m.index, "survey_year"].values, rownames=["value"], colnames=["year"])
    t.insert(0, "total", t.sum(axis=1))
    print("\n--- %s: values matched by %r ---" % (label, pattern))
    print(t.sort_values("total", ascending=False).head(25).to_string() if len(t) else "(nothing matched)")

for col in ["MainBranch", "Employment", "DevType", "OrgSize"]:
    print("%-11s filled: %d of %d | years with the column: %s" % (col, df[col].notna().sum(), len(df), sorted(df.loc[df[col].notna(), "survey_year"].unique())))
show_matches("MainBranch", CONFIG["MainBranch"]["drop"], "MainBranch DROP")
show_matches("Employment", CONFIG["Employment_student"], "Employment STUDENT")
show_matches("Employment", CONFIG["Employment_notworking"], "Employment NOT WORKING")
show_matches("DevType", CONFIG["DevType"]["drop"], "DevType DROP")
show_matches("OrgSize", CONFIG["OrgSize"]["drop"], "OrgSize DROP")
print("\nMainBranch values that are NOT dropped:")
print(explode_items("MainBranch")[~explode_items("MainBranch").str.contains(CONFIG["MainBranch"]["drop"], case=False, regex=True)].value_counts().to_string())
print("\nEmployment items that count as WORKING:")
ei = explode_items("Employment")
print(ei[ei.str.contains(CONFIG["Employment_working"], case=False, regex=True)].value_counts().to_string())
print("\nEmployment items that are neither working, student nor not-working (check these):")
rest = ei[~ei.str.contains("|".join([CONFIG["Employment_working"], CONFIG["Employment_student"], CONFIG["Employment_notworking"]]), case=False, regex=True)]
print(rest.value_counts().to_string() if len(rest) else "(none)")

MainBranch  filled: 283786 of 417467 | years with the column: [np.int64(2017), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Employment  filled: 400914 of 417467 | years with the column: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]
DevType     filled: 414546 of 417467 | years with the column: [np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]
OrgSize     filled: 377441 of 417467 | years with the column: [np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(202

In [18]:
mb_bad  = row_any("MainBranch", CONFIG["MainBranch"]["drop"])
dev_bad = row_any("DevType", CONFIG["DevType"]["drop"])
org_bad = row_any("OrgSize", CONFIG["OrgSize"]["drop"])
emp_filled = df["Employment"].notna()
emp_working = row_any("Employment", CONFIG["Employment_working"])
emp_student = row_any("Employment", CONFIG["Employment_student"])
emp_notwork = row_any("Employment", CONFIG["Employment_notworking"])
if STRICT_STUDENT:
    emp_bad = emp_filled & (emp_student | (~emp_working & emp_notwork))
else:
    emp_bad = emp_filled & ~emp_working & (emp_student | emp_notwork)          # answers like "prefer not to say" are unknown, not unemployed -> kept
if not STRICT_STUDENT:
    dev_bad = row_any("DevType", CONFIG["DevType"]["drop"].replace("student|", ""))
    mb_bad = row_any("MainBranch", CONFIG["MainBranch"]["drop"].replace("student|", ""))

rules = {"MainBranch student/learner/hobbyist": mb_bad, "Employment not working / student": emp_bad, "DevType student/unemployed/...": dev_bad, "OrgSize student/not working": org_bad}
first = pd.Series("", index=df.index)
for name, m in rules.items():                                   # attribute each dropped row to the first rule that hits it
    first = first.where(first != "", np.where(m, name, ""))
print("rows hit by each rule (a row can hit several):")
for name, m in rules.items():
    print("  %-40s %7d" % (name, int(m.sum())))
print("\nrows dropped (first rule that hits), by year:")
print(pd.crosstab(df["survey_year"], first.replace("", "kept")))
print("\nrows whose Employment cannot be checked (no Employment answer), by year:", df.loc[~emp_filled].groupby("survey_year").size().to_dict())

# consistency check: MainBranch says professional developer, but Employment says student / not working
prof = df["MainBranch"].fillna("").str.contains("developer by profession", case=False)
print("\nMainBranch = developer by profession but Employment not working/student:", int((prof & emp_bad).sum()))

before = len(df)
df = df.loc[first == ""].copy()
log_rows("3a students / hobbyists / unemployed", before, len(df))

rows hit by each rule (a row can hit several):
  MainBranch student/learner/hobbyist          677
  Employment not working / student           10882
  DevType student/unemployed/...             15872
  OrgSize student/not working                  427

rows dropped (first rule that hits), by year:
col_0        DevType student/unemployed/...  Employment not working / student  MainBranch student/learner/hobbyist  OrgSize student/not working   kept
survey_year                                                                                                                                           
2011                                     32                                 0                                    0                           87   2031
2012                                     64                                 0                                    0                          136   3540
2013                                     85                                 0                     

In [19]:
# no usable salary (this is the "no CompTotal" rule: rows rescued from ConvertedCompYearly are kept)
before = len(df)
print("comp_source among the remaining rows:", df["comp_source"].value_counts().to_dict())
df = df.loc[df["comp_source"] != "missing"].copy()
log_rows("3b no usable salary", before, len(df))

comp_source among the remaining rows: {'survey_usd': 366677, 'parsed_fx': 15238, 'rescued_usd': 12940}
[ROWS] 3b no usable salary                                   394855 ->  394855  (removed 0)


## 4. Cutoff for implausible salaries
Your `describe()` of the USD column (before this filter): p5 = 5,000, median 59,642, p95 = 220,000, **p99 = 1,000,000, p99.9 = 4,000,000, max 74 million**. The top end is noise (pence, hourly pay, extra zeros), and the annualising step multiplies that noise by 12 or 52. The floor must stay below 5,000 because `<$10,000` (5,000) and `<$20,000` (10,000) band midpoints are real answers.

Cell 4.1 shows what each candidate window would remove, per year and per source. Cell 4.2 applies two rules:

1. **Absolute window** `LOW`..`HIGH` (defaults 1,000 and 1,000,000). Below 1,000 USD a year is not a developer salary anywhere; above 1M the share of genuine answers is tiny next to the share of typos.
2. **Within country-year outlier flag** (`log10` salary more than `MAD_K` robust deviations from that country's median for that year, only for groups with at least 30 rows). This is a **flag**, not a drop, because a genuinely high-paid developer in a small market is not an error; filter on it in the model if you want.

In [20]:
s = df["Comp_USD_Annual"]
print(s.describe(percentiles=[.001, .01, .05, .25, .5, .75, .95, .99, .999]).round(0))
cands = [(1_000, 1_000_000), (1_000, 500_000), (2_000, 500_000), (5_000, 400_000), (10_000, 300_000)]
print("\nrows outside each candidate window (share of the remaining %d rows):" % len(df))
for lo, hi in cands:
    out = (s < lo) | (s > hi)
    print("  [%7s, %9s]  below: %6d  above: %6d  total: %6d  (%.2f%%)" % (format(lo, ","), format(hi, ","), int((s < lo).sum()), int((s > hi).sum()), int(out.sum()), 100 * out.mean()))
print("\nrows outside [1,000 ; 1,000,000] by year x source:")
bad = df.loc[(s < 1_000) | (s > 1_000_000)]
print(pd.crosstab(bad["survey_year"], bad["comp_source"]))
print("\nmedian and p99 by year:")
print(df.groupby("survey_year")["Comp_USD_Annual"].quantile([.5, .99]).unstack().round(0))

count      394855.0
mean       106114.0
std        428296.0
min             0.0
0.1%            0.0
1%            984.0
5%           5209.0
25%         30252.0
50%         62507.0
75%        105000.0
95%        225000.0
99%       1000000.0
99.9%     4200000.0
max      74351432.0
Name: Comp_USD_Annual, dtype: float64

rows outside each candidate window (share of the remaining 394855 rows):
  [  1,000, 1,000,000]  below:   4000  above:   3472  total:   7472  (1.89%)
  [  1,000,   500,000]  below:   4000  above:   9026  total:  13026  (3.30%)
  [  2,000,   500,000]  below:   6238  above:   9026  total:  15264  (3.87%)
  [  5,000,   400,000]  below:  14321  above:  10539  total:  24860  (6.30%)
  [ 10,000,   300,000]  below:  32938  above:  13201  total:  46139  (11.69%)

rows outside [1,000 ; 1,000,000] by year x source:
comp_source  rescued_usd  survey_usd
survey_year                         
2017                 318           0
2018                   2         671
2019                  

In [21]:
LOW, HIGH, MAD_K, MIN_GROUP = 1_000, 1_000_000, 4.0, 30

before = len(df)
df = df.loc[df["Comp_USD_Annual"].between(LOW, HIGH)].copy()
log_rows("4a salary outside [%s, %s] USD" % (format(LOW, ","), format(HIGH, ",")), before, len(df))

lg = np.log10(df["Comp_USD_Annual"])
grp = lg.groupby([df["survey_year"], df["Country"]])
med, n = grp.transform("median"), grp.transform("size")
mad = (lg - med).abs().groupby([df["survey_year"], df["Country"]]).transform("median") * 1.4826
df["comp_outlier_country"] = ((n >= MIN_GROUP) & (mad > 0) & ((lg - med).abs() > MAD_K * mad)).astype(int)
print("flagged as country-year outliers (kept, flag only):", int(df["comp_outlier_country"].sum()), "(%.2f%%)" % (100 * df["comp_outlier_country"].mean()))
print(df.groupby("survey_year")["comp_outlier_country"].mean().round(4).to_dict())

[ROWS] 4a salary outside [1,000, 1,000,000] USD              394855 ->  387383  (removed 7472)
flagged as country-year outliers (kept, flag only): 8740 (2.26%)
{2011: 0.0108, 2012: 0.0105, 2013: 0.0112, 2014: 0.01, 2015: 0.0063, 2016: 0.0181, 2017: 0.0243, 2018: 0.0338, 2019: 0.0372, 2020: 0.0303, 2021: 0.0192, 2022: 0.0195, 2023: 0.0147, 2024: 0.014, 2025: 0.018, 2026: 0.022}


## 5. Text columns: split on `;`, normalise, collapse spelling variants
**What I did with "collapse and split by semicolon":** every list-type column is kept as one normalised `;` string (one-hot comes later, after pooling, as you decided), with these steps per item: trim spaces, drop empty items, remove duplicates inside a row, sort alphabetically (so `A;B` and `B;A` are the same). **Collapse** = items that are identical once case, spacing and curly/straight quotes are ignored are merged into their most frequent spelling. Symbols are kept in the comparison, so `C`, `C#` and `C++` stay three items. Anything that differs in *wording* (e.g. two different phrasings of the same education level across years) is **not** merged automatically: the vocabulary files list item x year so you can add it to `COLLAPSE` yourself.

`LIST_COLS` are the multi-answer columns in your keep list; the single-answer text columns only get trimmed and get the same vocabulary report.

In [22]:
LIST_COLS = ["LearnCode", "DevType", "EdLevel", "Employment", "MainBranch", "LanguageHaveWorkedWith", "LanguageWantToWorkWith"]
SINGLE_TEXT = ["Country", "Age", "YearsCode", "YearsCodePro", "OrgSize", "Industry", "RemoteWork", "JobSat", "OpSys", "PurchaseInfluence"]
COLLAPSE = {   # column -> {variant: canonical}; add your own after reading the vocab_*.csv files
    # "EdLevel": {"Some college/university study without earning a degree": "Some college"},
}

def norm_key(x):
    # ignore case, spacing and curly vs straight quotes ONLY. Symbols stay: "C", "C#", "C++" and "F#" must remain different items.
    return re.sub(r"\s+", "", x.lower().replace("\u2019", "'").replace("\u2018", "'"))

def normalise_list_col(col):
    s = df[col]
    ex = s.dropna().astype(str).str.split(";").explode().str.strip()
    ex = ex[ex != ""]
    freq = ex.value_counts()
    # canonical spelling per normalised key = the most frequent raw spelling
    canon = {}
    for item in freq.index:                       # freq is sorted by count, so the first one seen wins
        canon.setdefault(norm_key(item), item)
    manual = COLLAPSE.get(col, {})
    def fix(item):
        item = manual.get(item, item)
        return canon.get(norm_key(item), item)
    merged = {i: fix(i) for i in freq.index if fix(i) != i}
    def clean(cell):
        if pd.isna(cell):
            return np.nan
        items = sorted({fix(p.strip()) for p in str(cell).split(";") if p.strip()}, key=str.lower)
        return ";".join(items) if items else np.nan
    out = s.map(clean)
    ex2 = out.dropna().str.split(";").explode()
    vocab = pd.crosstab(ex2.values, df.loc[ex2.index, "survey_year"].values, rownames=["item"], colnames=["year"])
    vocab.insert(0, "total", vocab.sum(axis=1))
    vocab.sort_values("total", ascending=False).to_csv("vocab_%s.csv" % col)
    print("%-26s items before: %4d | after: %4d | spelling variants merged: %3d | blank rows: %d" % (col, len(freq), len(vocab), len(merged), int(out.isna().sum())))
    for a, b in list(merged.items())[:5]:
        print("      merged %r -> %r" % (a, b))
    return out

for col in LIST_COLS:
    if col in df.columns:
        df[col] = normalise_list_col(col)
df["LanguageHaveWorkedWith_count"] = df["LanguageHaveWorkedWith"].str.split(";").str.len().astype("Int64")

for col in SINGLE_TEXT:
    if col in df.columns:
        df[col] = df[col].str.strip().replace("", np.nan)
        v = pd.crosstab(df[col], df["survey_year"]); v.insert(0, "total", v.sum(axis=1))
        v.sort_values("total", ascending=False).to_csv("vocab_%s.csv" % col)
        print("%-18s distinct values: %4d (item x year table in vocab_%s.csv)" % (col, df[col].nunique(), col))
print("\nafter the filters, are any students / hobbyists / unemployed left?")
for col, pat in [("MainBranch", CONFIG["MainBranch"]["drop"]), ("DevType", CONFIG["DevType"]["drop"])]:
    print("  %-11s items matching %r: %d" % (col, pat, int(explode_items(col).str.contains(pat, case=False, regex=True).sum())))

LearnCode                  items before:   61 | after:   61 | spelling variants merged:   0 | blank rows: 66117
DevType                    items before:  115 | after:  107 | spelling variants merged:   8 | blank rows: 2702
      merged 'Full-Stack Web Developer' -> 'Full-stack web developer'
      merged 'other' -> 'Other'
      merged 'Back-End Web Developer' -> 'Back-end web developer'
      merged 'Embedded Application Developer' -> 'Embedded application developer'
      merged 'Front-End Web Developer' -> 'Front-end web developer'
EdLevel                    items before:   23 | after:   23 | spelling variants merged:   0 | blank rows: 67622
Employment                 items before:   14 | after:   14 | spelling variants merged:   0 | blank rows: 15743
MainBranch                 items before:    6 | after:    6 | spelling variants merged:   0 | blank rows: 119470
LanguageHaveWorkedWith     items before:  161 | after:  157 | spelling variants merged:   4 | blank rows: 10012
      merg

## 6. Keep columns and save
Your list plus what the pipeline needs: `survey_year`, `uid` (your assert needs them, `ResponseId` repeats across years), `LanguageHaveWorkedWith_count`, and the new salary columns. **For modelling, `Comp_USD_Annual` is the target. `CompTotal`, `CompFreq`, `Currency` and `ConvertedCompYearly` are kept only so the numbers can be traced; they are the target in another form, so leave them out of the features.**

In [23]:
cols_to_keep = ["survey_year", "uid", "ResponseId", "MainBranch", "Country", "Age", "EdLevel", "LearnCode",
                "YearsCode", "YearsCodePro", "WorkExp", "Employment", "DevType", "OrgSize",
                "Industry", "RemoteWork", "JobSat", "CompTotal", "Currency", "CompFreq",
                "ConvertedCompYearly", "LanguageHaveWorkedWith", "LanguageHaveWorkedWith_count", "LanguageWantToWorkWith", "OpSys",
                "PurchaseInfluence", "PurchaseInfluence__InfluenceInternet", "PurchaseInfluence__InfluenceWorkstation",
                "PurchaseInfluence__InfluenceHardware", "PurchaseInfluence__InfluenceServers", "PurchaseInfluence__InfluenceTechStack",
                "PurchaseInfluence__InfluenceDeptTech", "PurchaseInfluence__InfluenceVizTools", "PurchaseInfluence__InfluenceDatabase",
                "PurchaseInfluence__InfluenceCloud", "PurchaseInfluence__InfluenceConsultants", "PurchaseInfluence__InfluenceRecruitment",
                "PurchaseInfluence__InfluenceCommunication",
                # added by this notebook
                "Currency_ISO", "CompIsTotalComp", "comp_source", "Comp_USD_Annual", "comp_outlier_country"]
missing_cols = [c for c in cols_to_keep if c not in df.columns]
print("not in the dataframe, skipped:", missing_cols)
df = df[[c for c in cols_to_keep if c in df.columns]].copy()
print("final shape:", df.shape)
print("\nrow log:")
for step, b, a in row_log:
    print("  %-52s %7d -> %7d" % (step, b, a))

not in the dataframe, skipped: []
final shape: (387383, 43)

row log:
  loaded                                                417467 ->  417467
  3a students / hobbyists / unemployed                  417467 ->  394855
  3b no usable salary                                   394855 ->  394855
  4a salary outside [1,000, 1,000,000] USD              394855 ->  387383


## THIS CELL NEEDS TO BE RUN AT THE VERY END
Checks that must pass before saving; writes `combined/all_years_fixed.csv`.

In [24]:
assert df["uid"].duplicated().sum() == 0, "duplicate uids"
assert df["survey_year"].notna().all(), "blank survey_year"
assert df["Comp_USD_Annual"].notna().all(), "a kept row has no salary"
assert df["Comp_USD_Annual"].between(LOW, HIGH).all(), "salary outside the cutoff window"
assert not explode_items("MainBranch").str.contains(CONFIG["MainBranch"]["drop"], case=False, regex=True).any(), "student/hobbyist left in MainBranch"
assert not explode_items("DevType").str.contains(CONFIG["DevType"]["drop"], case=False, regex=True).any(), "student/unemployed left in DevType"

df.to_csv("combined/all_years_fixed.csv", index=False)
print("saved", df.shape)
print(df.groupby("survey_year")["Comp_USD_Annual"].agg(["count", "median"]).round(0).T.to_string())

saved (387383, 43)
survey_year     2011     2012     2013     2014     2015     2016     2017     2018     2019     2020     2021     2022     2023     2024     2025     2026
count         2031.0   3540.0   5163.0   4504.0  13548.0  36966.0  12573.0  41561.0  50402.0  33996.0  44439.0  36345.0  47226.0  21313.0  21619.0  12157.0
median       70000.0  50000.0  70000.0  50000.0  50000.0  45000.0  51312.0  60000.0  59000.0  54000.0  57267.0  67233.0  75000.0  70000.0  81210.0  85096.0
